## Utility Functions

The function `range` is *overloaded*:
 * The function `range(start, stop)` creates the list `[start, ..., stop]`.
 * The function `range(stop)` creates the list `[0, ..., stop]`.

In [ ]:
%maven tools.aqua:z3-turnkey:4.14.1
import com.microsoft.z3.*;
import java.util.stream.*;

In [ ]:
List<Integer> range(int start, int stop) {
    return IntStream.rangeClosed(start, stop).boxed().toList();
}

List<Integer> range(int stop) {
    return range(0, stop);
}

In [ ]:
range(3)

In [ ]:
range(2, 5)

# The Knight's Tour

This notebook computes a solution to the [knight's tour](https://en.wikipedia.org/wiki/Knight%27s_tour) using the constraint solver `Z3`.  

All `Z3` objects are created by a *context*, which we call `Z3`.

In [ ]:
Context Z3 = new Context();

Given an integer from the set $\{0, 1, \cdots, 63\}$, the function `row(i)` computes the name of the variable that specifies the *row* of the knight after its $i^{\textrm{th}}$ move.

In [ ]:
String row(int i) {
    return "R" + i;
}

In [ ]:
row(1)

Given an integer from the set $\{0, 1, \cdots, 63\}$, the function `col(i)` computes the name of the variable that specifies the *column* of the knight after its $i^{\textrm{th}}$ move.

In [ ]:
String col(int i) {
    return "C" + i;
}

The function `toBitVec(x)` turns the number `x` into a *bit vector* of length 4.

In [ ]:
BitVecNum toBitVec(int x) {
    return Z3.mkBV(x, 4);
}

The function `toBitVar(x)` turns a string into a variable that has a type of bit vector of length 4.

In [ ]:
BitVecExpr toBitVar(String x) {
    return Z3.mkBVConst(x, 4);
}

The function `isKnightMove(row, col, rowX, colX)` takes four arguments:
* `row` is a `Z3` variable that specifies the row of the position of the knight before the move.
* `col` is a `Z3` variable that specifies the column of the position of the knight before the move.
* `rowX` is a `Z3` variable that specifies the row of the position of the knight after the move.
* `colX` is a `Z3` variable that specifies the column of the position of the knight after the move.

It returns a formula that specifies that the specified position represents a legal move for a knight.

In order to form the *conjunction* of two formulas we use the method `Z3.mkAnd`,
while the *disjunction* is build with the method `Z3.mkOr`.
Note that these methods can be called with any number of arguments.  The method `Z3.mkBVAdd` adds two bit vectors.

The figure below shows the moves of a knight:  The knight on `e4` can jump to all red squares.
<img src="knight-moves.png" width="50%">

In [ ]:
BoolExpr isKnightMove(BitVecExpr row, BitVecExpr col, BitVecExpr rowX, BitVecExpr colX) {
    int[][] deltaSet = { {1, 2}, {1, -2}, {2, 1}, {2, -1}, {-1, 2}, {-1, -2}, {-2, 1}, {-2, -1} };
    List<BoolExpr> formulas = new ArrayList<>();
    for (int[] delta : deltaSet) {
        formulas.add(Z3.mkAnd(Z3.mkEq(rowX, Z3.mkBVAdd(row, toBitVec(delta[0]))),     // rowX == row + deltaRow
                              Z3.mkEq(colX, Z3.mkBVAdd(col, toBitVec(delta[1])))));   // colX == col + deltaCol
    }
    return Z3.mkOr(formulas.toArray(BoolExpr[]::new));
}

The function `allDifferent` takes two arguments:
* `Rows` is a list of `Z3` variables. The variable `Rows[i]` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `Cols` is a list of `Z3` variables. The variable `Cols[i]` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

The function returns a set of formulas stating that for $i \not= j$ the positions after the $i^{\textrm{th}}$ move
differs from the position after the $j^{\textrm{th}}$ move.

In [ ]:
List<BoolExpr> allDifferent(List<BitVecExpr> Rows, List<BitVecExpr> Cols) {
    List<BoolExpr> result = new ArrayList<>();
    for (int i : range(62)) {
        for (int j : range(i + 1, 63)) {   // [Ri, Ci] != [Rj, Cj]  f.a. i < j
            result.add(Z3.mkOr(Z3.mkNot(Z3.mkEq(Rows.get(i), Rows.get(j))),
                               Z3.mkNot(Z3.mkEq(Cols.get(i), Cols.get(j)))));
        }
    }
    return result;
}

The function `allConstraints` takes two arguments:
* `Rows` is a list of `Z3` variables. The variable `Rows[i]` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `Cols` is a list of `Z3` variables. The variable `Cols[i]` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

`allConstraints` returns a set containing all constraints of the problem.

The method `Z3.mkBVULT` compares two bit vectors that are interpreted as *unsigned* numbers.

In [ ]:
List<BoolExpr> allConstraints(List<BitVecExpr> Rows, List<BitVecExpr> Cols) {
    var zero  = toBitVec(0);
    var eight = toBitVec(8);
    List<BoolExpr> result = new ArrayList<>(allDifferent(Rows, Cols));
    result.add(Z3.mkEq(Rows.get(0), zero));
    result.add(Z3.mkEq(Cols.get(0), zero));
    for (int i : range(62)) {
        result.add(isKnightMove(Rows.get(i), Cols.get(i), Rows.get(i + 1), Cols.get(i + 1)));
    }
    for (int i : range(63)) {
        result.add(Z3.mkBVULT(Rows.get(i), eight));
        result.add(Z3.mkBVULT(Cols.get(i), eight));
    }
    return result;
}

The function `parseVal` takes a bit vector that is a value of a model and converts it into an integer.

In [ ]:
int parseVal(Expr<?> expr) {
    return ((BitVecNum) expr).getInt();
}

The function `solve()` computes a solution of the knight's problem and returns this solution as a list of positions.  Every position is a list of the form `[row, col]`.

In [ ]:
List<List<Integer>> solve() {
    List<BitVecExpr> Rows = range(63).stream().map(i -> toBitVar(row(i))).toList();
    List<BitVecExpr> Cols = range(63).stream().map(i -> toBitVar(col(i))).toList();
    Solver solver = Z3.mkSolver();
    solver.add(allConstraints(Rows, Cols).toArray(BoolExpr[]::new));
    var check = solver.check();
    if (check == Status.SATISFIABLE) {
        Model model = solver.getModel();
        return range(63).stream()
                        .map(i -> List.of(parseVal(model.eval(Rows.get(i), true)),
                                          parseVal(model.eval(Cols.get(i), true))))
                        .toList();
    } else if (check == Status.UNSATISFIABLE) {
        System.out.println("The problem is not solvable.");
    } else {
        System.out.println("Z3 cannot determine whether the problem is solvable.");
    }
    return null;
}

Unfortunately, the execution time of the following cell varies greatly between different runs.  On my computer, it took about three minutes.

In [ ]:
long start   = System.currentTimeMillis();
var Solution = solve();
System.out.println("Solver Duration: " + (System.currentTimeMillis() - start) / 1000.0 + " s");
Solution

The function `createBoard(Solution)` returns a matrix `Board` of size $8\times 8$.
The following holds:
$$ \texttt{Board}[\texttt{R}i][\texttt{C}i] = i $$
Therefore, if `Board[r][c] == i`, then at the beginning of the $i^{\textrm{th}}$ move the knight is located in row `r` and column `c`. 

In [ ]:
int[][] createBoard(List<List<Integer>> solution) {
    int[][] board = new int[8][8];
    for (int i = 0; i < solution.size(); i++) {
        board[solution.get(i).get(0)][solution.get(i).get(1)] = i;
    }
    return board;
}

The function `Arrays.deepToString` converts a two-dimensional array into a string.

In [ ]:
Arrays.deepToString(createBoard(Solution))

The function `printBoard` prints the given `Board`.

In [ ]:
// createLine(left, mid, right, fill, n, width) creates a horizontal line of the table
String createLine(String left, String mid, String right, String fill, int n, int width) {
    var line = new StringBuilder(left);
    for (int i = 0; i < n - 1; i++) {
        line.append(fill.repeat(width + 2)).append(mid);
    }
    line.append(fill.repeat(width + 2)).append(right);
    return line.toString();
}

void printBoard(int[][] board) {
    int n = board.length;
    if (n == 0) {
        return;
    }
    int width = Arrays.stream(board)
                      .flatMapToInt(Arrays::stream)
                      .map(element -> String.valueOf(element).length())
                      .max().getAsInt();
    String topLine = createLine("╔", "╦", "╗", "═", n, width);
    String midLine = createLine("╠", "╬", "╣", "═", n, width);
    String botLine = createLine("╚", "╩", "╝", "═", n, width);
    System.out.println(topLine);
    for (int i = 0; i < n; i++) {
        var line = new StringBuilder("║");
        for (int element : board[i]) {
            line.append(" ").append(String.format("%" + width + "d", element)).append(" ║");
        }
        System.out.println(line);
        if (i < n - 1) {
            System.out.println(midLine);
        }
    }
    System.out.println(botLine);
}

In [ ]:
var board = createBoard(Solution);

In [ ]:
printBoard(board)

# Visualization

The function `showSolution` displays the given solution on a chessboard.
The solution `Board` is represented as a list of lists.  We have `Board[row][col] == k` if the $k^\textrm{th}$ move leads the knight to the position `(row, col)`.

In [ ]:
void showSolution(int[][] board, String width) {
    int n         = board.length;
    int cellSize  = 50;
    int totalSize = n * cellSize;
    int[][] path  = new int[n * n][];
    for (int r = 0; r < n; r++) {
        for (int c = 0; c < n; c++) {
            int k = board[r][c];
            if (k >= 0 && k < n * n) {
                path[k] = new int[] { r, c };
            }
        }
    }
    String uniqueId = Integer.toString(new Random().nextInt(Integer.MAX_VALUE), 36);
    var html = new StringBuilder();
    html.append("<div id=\"tour-container-" + uniqueId + "\">");
    html.append("<button id=\"btn-" + uniqueId + "\" onclick=\"playKnightTour_" + uniqueId + "()\" "
              + "style=\"padding: 10px 15px; margin-bottom: 10px; cursor: pointer; background: #4CAF50; "
              + "color: white; border: none; border-radius: 4px; font-size: 14px;\">▶ Play Epic Tour</button><br>");
    html.append("<svg width=\"" + width + "\" viewBox=\"0 0 " + totalSize + " " + totalSize
              + "\" xmlns=\"http://www.w3.org/2000/svg\" style=\"font-family: sans-serif;\">");
    // Draw board
    for (int r = 0; r < n; r++) {
        for (int c = 0; c < n; c++) {
            boolean isBlack = (r + c) % 2 != 0;
            String  color   = isBlack ? "#b58863" : "#f0d9b5";
            int x = c * cellSize;
            int y = r * cellSize;
            html.append("<rect x=\"%d\" y=\"%d\" width=\"%d\" height=\"%d\" fill=\"%s\" />"
                        .formatted(x, y, cellSize, cellSize, color));
            html.append("<text x=\"%d\" y=\"%d\" text-anchor=\"middle\" font-size=\"14\" fill=\"%s\">%d</text>"
                        .formatted(x + cellSize / 2, y + cellSize / 2 + 5, isBlack ? "#f0d9b5" : "#b58863", board[r][c]));
        }
    }
    // Start position
    if (path[0] != null) {
        html.append("<circle cx=\"%d\" cy=\"%d\" r=\"8\" fill=\"darkblue\" />"
                    .formatted(path[0][1] * cellSize + cellSize / 2, path[0][0] * cellSize + cellSize / 2));
    }
    // Path lines and dots
    for (int k = 0; k < n * n - 1; k++) {
        int[] p1 = path[k];
        int[] p2 = path[k + 1];
        if (p1 != null && p2 != null) {
            int x1 = p1[1] * cellSize + cellSize / 2;
            int y1 = p1[0] * cellSize + cellSize / 2;
            int x2 = p2[1] * cellSize + cellSize / 2;
            int y2 = p2[0] * cellSize + cellSize / 2;
            html.append(("<line id=\"line-%s-%d\" x1=\"%d\" y1=\"%d\" x2=\"%d\" y2=\"%d\" stroke=\"darkblue\" "
                       + "stroke-width=\"3\" stroke-opacity=\"0.6\" opacity=\"0\"></line>")
                        .formatted(uniqueId, k, x1, y1, x2, y2));
            html.append("<circle id=\"circle-%s-%d\" cx=\"%d\" cy=\"%d\" r=\"3\" fill=\"darkblue\" opacity=\"0\"></circle>"
                        .formatted(uniqueId, k, x2, y2));
        }
    }
    html.append("</svg>");
    // Animation and Audio Logic
    html.append("""
    <script>
        let isPlaying_@uniqueId@ = false;

        function playKnightTour_@uniqueId@() {
            if (isPlaying_@uniqueId@) return;
            isPlaying_@uniqueId@ = true;
            
            const btn = document.getElementById('btn-@uniqueId@');
            btn.style.background = '#888';
            btn.innerText = 'Running...';

            const maxSteps = @n*n-1@;
            const delay = 500; // Half speed (was 250ms)
            
            const AudioContext = window.AudioContext || window.webkitAudioContext;
            const audioCtx = new AudioContext();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            // --- 1. SETUP REVERB ---
            const convolver = audioCtx.createConvolver();
            const rate = audioCtx.sampleRate;
            const length = rate * 2.0; // 2 seconds of reverb decay
            const impulse = audioCtx.createBuffer(2, length, rate);
            for (let i = 0; i < length; i++) {
                const decay = Math.exp(-i / (rate * 0.3)); // Reverb tail
                impulse.getChannelData(0)[i] = (Math.random() * 2 - 1) * decay;
                impulse.getChannelData(1)[i] = (Math.random() * 2 - 1) * decay;
            }
            convolver.buffer = impulse;
            convolver.connect(audioCtx.destination);

            // --- 2. SOUND DESIGN FUNCTIONS ---
            function playNeigh() {
                const osc = audioCtx.createOscillator();
                const lfo = audioCtx.createOscillator();
                const tremolo = audioCtx.createGain();
                const mainGain = audioCtx.createGain();
                
                // The base sound: A descending sawtooth
                osc.type = 'sawtooth';
                osc.frequency.setValueAtTime(900, audioCtx.currentTime);
                osc.frequency.exponentialRampToValueAtTime(300, audioCtx.currentTime + 1.2);
                
                // Tremolo effect to simulate the "chatter" of a whinny
                lfo.type = 'sine';
                lfo.frequency.value = 18; 
                tremolo.gain.value = 0.8;
                
                mainGain.gain.setValueAtTime(0, audioCtx.currentTime);
                mainGain.gain.linearRampToValueAtTime(0.5, audioCtx.currentTime + 0.1);
                mainGain.gain.setValueAtTime(0.5, audioCtx.currentTime + 0.8);
                mainGain.gain.exponentialRampToValueAtTime(0.01, audioCtx.currentTime + 1.2);

                lfo.connect(tremolo.gain);
                osc.connect(tremolo);
                tremolo.connect(mainGain);
                
                // Route to both direct output and reverb
                mainGain.connect(audioCtx.destination);
                mainGain.connect(convolver);
                
                lfo.start(); osc.start();
                lfo.stop(audioCtx.currentTime + 1.3); osc.stop(audioCtx.currentTime + 1.3);
            }

            function playMoveSound() {
                const osc = audioCtx.createOscillator();
                const gainNode = audioCtx.createGain();
                
                osc.type = 'sine';
                osc.frequency.setValueAtTime(300, audioCtx.currentTime);
                osc.frequency.exponentialRampToValueAtTime(500, audioCtx.currentTime + 0.05);
                
                gainNode.gain.setValueAtTime(0.4, audioCtx.currentTime);
                gainNode.gain.exponentialRampToValueAtTime(0.001, audioCtx.currentTime + 0.05);
                
                osc.connect(gainNode);
                
                // Route to direct output AND reverb to create the echo
                gainNode.connect(audioCtx.destination);
                gainNode.connect(convolver);
                
                osc.start();
                osc.stop(audioCtx.currentTime + 0.05);
            }

            function playVictorySound() {
                const notes = [261.63, 329.63, 392.00, 523.25]; // C4, E4, G4, C5 Fanfare
                const startTime = audioCtx.currentTime;
                
                notes.forEach((freq, index) => {
                    const osc = audioCtx.createOscillator();
                    const gain = audioCtx.createGain();
                    osc.type = 'triangle';
                    osc.frequency.value = freq;
                    
                    const time = startTime + index * 0.15;
                    const duration = index === notes.length - 1 ? 2.0 : 0.15;
                    
                    gain.gain.setValueAtTime(0, time);
                    gain.gain.linearRampToValueAtTime(0.3, time + 0.05);
                    gain.gain.exponentialRampToValueAtTime(0.01, time + duration);
                    
                    osc.connect(gain);
                    gain.connect(audioCtx.destination);
                    gain.connect(convolver);
                    
                    osc.start(time);
                    osc.stop(time + duration);
                });
            }

            // --- 3. ANIMATION SEQUENCE ---
            // Reset visuals
            for (let i = 0; i < maxSteps; i++) {
                const line = document.getElementById('line-@uniqueId@-' + i);
                const circle = document.getElementById('circle-@uniqueId@-' + i);
                if (line) line.setAttribute('opacity', '0');
                if (circle) circle.setAttribute('opacity', '0');
            }

            // Play the opening neigh, then wait 1.5 seconds before starting the tour
            playNeigh();
            
            setTimeout(() => {
                let currentStep = 0;
                const intervalId = setInterval(() => {
                    if (currentStep >= maxSteps) {
                        clearInterval(intervalId);
                        playVictorySound();
                        btn.style.background = '#4CAF50';
                        btn.innerText = '▶ Replay Tour';
                        isPlaying_@uniqueId@ = false;
                        return;
                    }
                    
                    // Reveal SVG paths
                    const line = document.getElementById('line-@uniqueId@-' + currentStep);
                    const circle = document.getElementById('circle-@uniqueId@-' + currentStep);
                    if (line) line.setAttribute('opacity', '1');
                    if (circle) circle.setAttribute('opacity', '1');
                    
                    playMoveSound();
                    currentStep++;
                    
                }, delay);
            }, 1500); // 1.5 second pause for the horse neigh
        }
    </script>
    </div>
        """.replace("@uniqueId@", uniqueId)
           .replace("@n*n-1@", String.valueOf(n * n - 1)));
    display(html.toString(), "text/html");
}

void showSolution(int[][] board) {
    showSolution(board, "50%");
}

In [ ]:
showSolution(board);